# X Personality Analyzer

Give it any X/Twitter username, it scrapes their posts and uses Gemini AI to build a personality report.

**Setup:** Get a free Gemini API key at [aistudio.google.com/apikey](https://aistudio.google.com/apikey), paste it in the config cell, and run all cells.

In [ ]:
# install dependencies
!pip install -q requests google-genai rich playwright
!playwright install chromium 2>/dev/null || echo "already installed"

In [20]:
# imports
import asyncio, json, os, re, time
from datetime import datetime
from pathlib import Path

import requests
from google import genai
from google.genai import types
from playwright.async_api import async_playwright, TimeoutError as PlaywrightTimeout
from rich.console import Console
from rich.panel import Panel
from rich.table import Table

console = Console()
COOKIE_FILE = Path.cwd() / ".x_cookies.json"

print("Ready!")
print(f"Cookies: {'found' if COOKIE_FILE.exists() else 'not found (run login cell first)'}")

Ready!
Cookies: found


In [21]:
# login to X (run once, then skip)
# opens a browser — log in manually, cookies get saved for scraping

async def login_to_x():
    async with async_playwright() as pw:
        browser = await pw.chromium.launch(
            headless=False,
            args=["--disable-blink-features=AutomationControlled", "--no-sandbox"],
        )
        ctx = await browser.new_context(
            viewport={"width": 1280, "height": 900},
            user_agent="Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36",
        )
        await ctx.add_init_script("Object.defineProperty(navigator, 'webdriver', { get: () => undefined });")
        page = await ctx.new_page()
        await page.goto("https://x.com/i/flow/login", wait_until="domcontentloaded")

        print("Browser opened — log in to X. Cookies save automatically when you reach the home page.")

        try:
            await page.wait_for_url("**/home", timeout=300_000)
        except PlaywrightTimeout:
            await browser.close()
            raise RuntimeError("Timed out waiting for login.")

        await asyncio.sleep(3)
        cookies = await ctx.cookies()
        COOKIE_FILE.write_text(json.dumps(cookies, indent=2))
        await browser.close()

    print("Cookies saved!")

# uncomment to log in:
# await login_to_x()

In [22]:
# scraper — uses saved cookies to grab posts from any profile

UA = "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36"


async def fetch_posts_browser(username, max_posts=30):
    """scrape posts using browser with saved cookies"""
    username = username.lstrip("@")

    if not COOKIE_FILE.exists():
        raise RuntimeError("No cookies found. Run the login cell first.")

    pw = await async_playwright().start()
    browser = await pw.chromium.launch(
        headless=True,
        args=["--disable-blink-features=AutomationControlled", "--no-sandbox"],
    )
    ctx = await browser.new_context(viewport={"width": 1280, "height": 900}, user_agent=UA, locale="en-US")
    await ctx.add_init_script("Object.defineProperty(navigator, 'webdriver', { get: () => undefined });")

    cookies = json.loads(COOKIE_FILE.read_text())
    await ctx.add_cookies(cookies)
    page = await ctx.new_page()

    print(f"Loading @{username}'s profile...")

    # try loading the profile, retry once on timeout
    loaded = False
    for attempt in range(2):
        await page.goto(f"https://x.com/{username}", wait_until="domcontentloaded", timeout=60000)
        try:
            await page.wait_for_selector("article[data-testid='tweet']", timeout=30000)
            loaded = True
            break
        except PlaywrightTimeout:
            if attempt == 0:
                print("  Slow load, retrying...")
                await asyncio.sleep(3)

    if not loaded:
        body = await page.inner_text("body")
        await browser.close()
        await pw.stop()
        if "doesn't exist" in body.lower() or "suspended" in body.lower():
            raise ValueError(f"@{username} not found or suspended.")
        if "protected" in body.lower():
            raise ValueError(f"@{username} is protected — you need to follow them.")
        raise ValueError(f"No tweets loaded. Try logging in again (cookies may have expired).")

    await asyncio.sleep(2)

    posts = []
    seen = set()
    empty_rounds = 0

    while len(posts) < max_posts and empty_rounds < 8:
        articles = await page.query_selector_all("article[data-testid='tweet']")
        found = 0

        for el in articles:
            if len(posts) >= max_posts:
                break
            try:
                # get text
                text_el = await el.query_selector("[data-testid='tweetText']")
                text = (await text_el.inner_text()).strip() if text_el else ""
                if not text or text.startswith("RT @"):
                    continue

                # get date and url
                time_el = await el.query_selector("time")
                date = ""
                url = ""
                if time_el:
                    date = (await time_el.get_attribute("datetime")) or ""
                    if date:
                        try:
                            date = datetime.fromisoformat(date).strftime("%Y-%m-%d")
                        except ValueError:
                            pass
                    link = await time_el.evaluate_handle("el => el.closest('a')")
                    if link:
                        href = (await link.evaluate("el => el.getAttribute('href')")) or ""
                        if href:
                            url = f"https://x.com{href}" if href.startswith("/") else href

                if url in seen:
                    continue
                if url:
                    seen.add(url)

                # get likes/reposts/replies
                likes = reposts = replies = 0
                group = await el.query_selector("[role='group']")
                if group:
                    buttons = await group.query_selector_all("button")
                    keys = ["replies", "reposts", "likes"]
                    for i, btn in enumerate(buttons):
                        if i < len(keys):
                            aria = (await btn.get_attribute("aria-label")) or ""
                            nums = re.findall(r"[\d,]+", aria)
                            val = int(nums[0].replace(",", "")) if nums else 0
                            if keys[i] == "likes": likes = val
                            elif keys[i] == "reposts": reposts = val
                            elif keys[i] == "replies": replies = val

                posts.append({"text": text, "date": date, "likes": likes, "reposts": reposts, "replies": replies})
                found += 1
            except Exception:
                continue

        empty_rounds = empty_rounds + 1 if found == 0 else 0
        if len(posts) < max_posts:
            await page.evaluate("window.scrollBy(0, window.innerHeight * 2)")
            await asyncio.sleep(2.5)
        if len(posts) % 20 == 0 and len(posts) > 0:
            print(f"  {len(posts)} posts collected so far...")

    # save fresh cookies
    fresh = await ctx.cookies()
    COOKIE_FILE.write_text(json.dumps(fresh, indent=2))
    await browser.close()
    await pw.stop()

    return posts[:max_posts]


async def fetch_posts(username, max_posts=30):
    username = username.lstrip("@")
    print(f"Scraping @{username}...")
    return await fetch_posts_browser(username, max_posts)


print("Scraper ready!")

Scraper ready!


In [ ]:
# gemini AI analyzer
# put your API key here (free: https://aistudio.google.com/apikey)

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "YOUR_API_KEY_HERE")

# tries these models in order if one gets rate limited
GEMINI_MODELS = ["gemini-2.5-flash", "gemini-2.5-flash-lite", "gemini-2.0-flash-lite", "gemini-2.0-flash"]


def build_prompt(posts, username):
    lines = []
    for i, p in enumerate(posts, 1):
        meta = f"Date: {p['date']} | Likes: {p['likes']} | Reposts: {p['reposts']}"
        lines.append(f"[Post {i}] [{meta}]\n{p['text']}\n")
    posts_text = "\n".join(lines)

    return f"""You are an expert personality analyst. Analyze the following posts from @{username} and generate a comprehensive personality profile.

Posts:
---
{posts_text}
---

Respond with JSON ONLY (no extra text) following this exact structure:

{{
  "personality_summary": "A paragraph (5-8 sentences) summarizing the user's personality",
  "mbti_estimate": {{
    "type": "e.g. INFP",
    "confidence": "high/medium/low",
    "explanation": "Why this type fits"
  }},
  "big_five": {{
    "openness": {{"score": 0-100, "level": "High/Medium/Low", "evidence": "Evidence from posts"}},
    "conscientiousness": {{"score": 0-100, "level": "High/Medium/Low", "evidence": "Evidence"}},
    "extraversion": {{"score": 0-100, "level": "High/Medium/Low", "evidence": "Evidence"}},
    "agreeableness": {{"score": 0-100, "level": "High/Medium/Low", "evidence": "Evidence"}},
    "neuroticism": {{"score": 0-100, "level": "High/Medium/Low", "evidence": "Evidence"}}
  }},
  "communication_style": {{
    "tone": "Overall tone description",
    "formality": "Formal/Semi-formal/Informal",
    "humor_level": "High/Medium/Low",
    "emotional_expression": "How they express emotions",
    "vocabulary_richness": "Rich/Moderate/Simple"
  }},
  "interests_and_topics": [
    {{"topic": "Topic name", "intensity": "Passionate/Interested/Casual", "description": "Brief description"}}
  ],
  "emotional_profile": {{
    "dominant_emotions": ["Emotion1", "Emotion2", "Emotion3"],
    "emotional_range": "Wide/Moderate/Narrow",
    "positivity_ratio": 0-100,
    "description": "Description of emotional profile"
  }},
  "social_behavior": {{
    "interaction_style": "How they interact with others",
    "influence_type": "Thought leader/Active participant/Observer/Content creator",
    "community_engagement": "High/Medium/Low"
  }},
  "values_and_beliefs": ["Value1", "Value2", "Value3"],
  "strengths": ["Strength1", "Strength2", "Strength3"],
  "growth_areas": ["Area1", "Area2"],
  "fun_facts": ["Fun fact1", "Fun fact2", "Fun fact3"],
  "one_liner": "Describe the user in one creative sentence"
}}"""


def analyze_with_gemini(posts, username, api_key=None, max_retries=3):
    """send posts to gemini, get personality report back"""
    key = api_key or GEMINI_API_KEY
    if key == "YOUR_API_KEY_HERE":
        raise ValueError("No API key! Get one free at https://aistudio.google.com/apikey and paste it above.")
    client = genai.Client(api_key=key)
    prompt = build_prompt(posts, username)

    last_error = None
    for model in GEMINI_MODELS:
        print(f"Trying {model}...")
        for attempt in range(max_retries):
            try:
                response = client.models.generate_content(
                    model=model,
                    contents=prompt,
                    config=types.GenerateContentConfig(
                        temperature=0.7,
                        max_output_tokens=8192,
                        response_mime_type="application/json",
                    ),
                )
                raw = response.text.strip()
                if raw.startswith("```"):
                    raw = re.sub(r"^```(?:json)?\s*", "", raw)
                    raw = re.sub(r"\s*```$", "", raw)
                try:
                    return json.loads(raw)
                except json.JSONDecodeError:
                    # try to extract the outermost JSON object
                    depth = 0
                    start = raw.find("{")
                    if start != -1:
                        for i in range(start, len(raw)):
                            if raw[i] == "{": depth += 1
                            elif raw[i] == "}": depth -= 1
                            if depth == 0:
                                return json.loads(raw[start:i+1])
                    raise ValueError(f"Bad JSON from Gemini:\n{raw[:500]}")
            except Exception as e:
                last_error = e
                if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
                    if attempt < max_retries - 1:
                        wait = 30 * (attempt + 1)
                        print(f"  Rate limited, waiting {wait}s...")
                        time.sleep(wait)
                    else:
                        print(f"  {model} quota used up, trying next...")
                        break
                elif isinstance(e, (json.JSONDecodeError, ValueError)) and "Bad JSON" in str(e):
                    print(f"  Bad response, retrying...")
                    continue
                else:
                    raise

    raise RuntimeError(f"All models failed. Last error: {last_error}")


print("Analyzer ready!")

Analyzer ready!


In [24]:
# pretty print the report

def print_report(report, username):
    console.print()
    console.print(Panel(f"[bold white on blue]  Personality Report — @{username}  [/]", border_style="blue", expand=False))

    one_liner = report.get("one_liner", "")
    if one_liner:
        console.print(Panel(f"[italic]{one_liner}[/italic]", border_style="magenta", expand=False))

    summary = report.get("personality_summary", "")
    if summary:
        console.print(Panel(summary, title="[bold]Summary[/]", border_style="dim"))

    # MBTI
    mbti = report.get("mbti_estimate", {})
    if mbti:
        text = f"[bold magenta]{mbti.get('type', '?')}[/bold magenta]\nConfidence: {mbti.get('confidence', '?')}\n\n{mbti.get('explanation', '')}"
        console.print(Panel(text, title="[bold]MBTI[/]", border_style="magenta"))

    # Big Five
    big5 = report.get("big_five", {})
    if big5:
        def bar(v, w=25):
            filled = round(v / 100 * w)
            return "█" * filled + "░" * (w - filled)

        t = Table(show_header=True, box=None, padding=(0, 2))
        t.add_column("Trait", style="bold", min_width=20)
        t.add_column("Score", justify="right", width=6)
        t.add_column("Bar", min_width=28)

        for key in ["openness", "conscientiousness", "extraversion", "agreeableness", "neuroticism"]:
            trait = big5.get(key, {})
            score = trait.get("score", 0)
            c = "bold green" if score >= 65 else ("bold yellow" if score >= 35 else "bold red")
            t.add_row(key.title(), f"[{c}]{score}[/]", bar(score))

        console.print(Panel(t, title="[bold]Big Five[/]", border_style="dim"))

        for key in ["openness", "conscientiousness", "extraversion", "agreeableness", "neuroticism"]:
            evidence = big5.get(key, {}).get("evidence", "")
            if evidence:
                console.print(f"  [dim]{key.title()}:[/dim] {evidence}")
        console.print()

    # Communication style
    comm = report.get("communication_style", {})
    if comm:
        ct = Table(show_header=False, box=None, padding=(0, 2))
        ct.add_column(style="bold cyan", min_width=22)
        ct.add_column(style="white")
        for label, key in [("Tone", "tone"), ("Formality", "formality"), ("Humor", "humor_level"), ("Emotion", "emotional_expression"), ("Vocabulary", "vocabulary_richness")]:
            ct.add_row(label, comm.get(key, ""))
        console.print(Panel(ct, title="[bold]Communication Style[/]", border_style="dim"))

    # Interests
    interests = report.get("interests_and_topics", [])
    if interests:
        it = Table(show_header=True, box=None, padding=(0, 2))
        it.add_column("Topic", style="bold")
        it.add_column("Level", style="magenta")
        it.add_column("Description", style="dim", max_width=50)
        for item in interests:
            it.add_row(item.get("topic", ""), item.get("intensity", ""), item.get("description", ""))
        console.print(Panel(it, title="[bold]Interests[/]", border_style="dim"))

    # Emotions
    emo = report.get("emotional_profile", {})
    if emo:
        parts = []
        doms = emo.get("dominant_emotions", [])
        if doms:
            parts.append(f"Dominant: {', '.join(doms)}")
        parts.append(f"Range: {emo.get('emotional_range', '?')}")
        parts.append(f"Positivity: {emo.get('positivity_ratio', '?')}%")
        desc = emo.get("description", "")
        if desc:
            parts.append(f"\n{desc}")
        console.print(Panel("\n".join(parts), title="[bold]Emotions[/]", border_style="dim"))

    # Social
    social = report.get("social_behavior", {})
    if social:
        text = f"Style: {social.get('interaction_style', '?')}\nInfluence: {social.get('influence_type', '?')}\nEngagement: {social.get('community_engagement', '?')}"
        console.print(Panel(text, title="[bold]Social[/]", border_style="dim"))

    # Quick facts table
    values = report.get("values_and_beliefs", [])
    strengths = report.get("strengths", [])
    growth = report.get("growth_areas", [])
    fun = report.get("fun_facts", [])

    dt = Table(show_header=False, box=None, padding=(0, 2))
    dt.add_column(style="bold cyan", min_width=16)
    dt.add_column(style="white")
    if values: dt.add_row("Values", " · ".join(values))
    if strengths: dt.add_row("Strengths", " · ".join(strengths))
    if growth: dt.add_row("Growth Areas", " · ".join(growth))
    if fun: dt.add_row("Fun Facts", " · ".join(fun))
    console.print(Panel(dt, title="[bold]Details[/]", border_style="green"))

    console.print("\n[dim]Generated by Gemini AI · not a clinical assessment[/dim]\n")


print("Report printer ready!")

Report printer ready!


In [25]:
# run it!

username = input("Username (e.g. elonmusk): ").strip().lstrip("@")
max_posts = int(input("How many posts? (default 30): ").strip() or "30")

print(f"\nScraping @{username}...")
posts = await fetch_posts(username, max_posts=max_posts)
print(f"Got {len(posts)} posts!\n")

if posts:
    # quick preview
    preview = Table(title=f"@{username}", show_lines=True)
    preview.add_column("#", style="dim", width=4)
    preview.add_column("Date", style="green", width=12)
    preview.add_column("Post", style="white", max_width=70, overflow="fold")
    preview.add_column("Likes", style="red", justify="right", width=8)

    for i, p in enumerate(posts[:10], 1):
        text = p["text"][:150] + "..." if len(p["text"]) > 150 else p["text"]
        preview.add_row(str(i), p["date"], text, f"{p['likes']:,}")

    console.print(preview)
    if len(posts) > 10:
        print(f"\n... and {len(posts) - 10} more posts")

    print("\nAnalyzing personality...")
    report = analyze_with_gemini(posts, username)
    print_report(report, username)

    # save
    filename = f"{username}_personality.json"
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(report, f, ensure_ascii=False, indent=2)
    print(f"Saved to {filename}")
else:
    print("No posts found. Try running the login cell first.")


Scraping @meowiestgurl90...
Scraping @meowiestgurl90...
Loading @meowiestgurl90's profile...
  Slow load, retrying...
Got 142 posts!



                                              @meowiestgurl90                                              
┏━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┓
┃ #    ┃ Date         ┃ Post                                                                   ┃    Likes ┃
┡━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━┩
│ 1    │ 2026-03-04   │ احب رمضان اموت فيه ودي يتم تمديده                                      │        0 │
├──────┼──────────────┼────────────────────────────────────────────────────────────────────────┼──────────┤
│ 2    │ 2026-03-04   │ اللي قاعد يصير لي امتحان صبر                                           │        0 │
├──────┼──────────────┼────────────────────────────────────────────────────────────────────────┼──────────┤
│ 3    │ 2026-03-03   │ جاك مهووس بس تخصصه AI:                                                 │        3 │
├──────┼──────────────┼────────────────────────────────────────────────────────────────────────┼──────────┤
│ 4    │ 2026-03-03   │ أتمنى بعد ما تنتهي الأزمة يتم ترحيل كل شيعي وشيعية إلى إيران           │    3,592 │
├──────┼──────────────┼────────────────────────────────────────────────────────────────────────┼──────────┤
│ 5    │ 2026-03-02   │ استعيذوا بالله من انقلاب الحال وفواجع الأقدار                          │   11,185 │
│      │              │                                                                        │          │
│      │              │ اللهم إنا نعوذ من الفواجع والصدمات ونعوذ بك من انقلاب الحال لشيء لا    │          │
│      │              │ يسر اللهم لا تفجعنا في أهلنا وأحباب...                                 │          │
├──────┼──────────────┼────────────────────────────────────────────────────────────────────────┼──────────┤
│ 6    │ 2026-03-03   │ Solo leveling                                                          │        2 │
│      │              │ محظوظ اللي ما تابعه قسم بالله عالممم ثانني                             │          │
│      │              │ وش اقوول الاوست حقهم ناررررر                                           │          │
├──────┼──────────────┼────────────────────────────────────────────────────────────────────────┼──────────┤
│ 7    │ 2026-03-02   │ والله هالبحث ماصار بحث يارب ينشرونه وخلاص                              │        1 │
├──────┼──────────────┼────────────────────────────────────────────────────────────────────────┼──────────┤
│ 8    │ 2026-03-01   │ كنت اشوفك مثل نجمً في السماء ساطع وعالي                                 │        0 │
│      │              │ يا بعيد وجابك الله لين عندي وصلك                                       │          │
│      │              │ جيتني مثل الشروق اللي محى عتم الليالي                                  │          │
│      │              │ جيت فرحه للحزين اللي من همومه هلك                                      │          │
│      │              │ جيتني...                                                               │          │
├──────┼──────────────┼────────────────────────────────────────────────────────────────────────┼──────────┤
│ 9    │ 2026-03-01   │ بتفهموني الحين ليش حطيتها ان شاء الله                                  │        3 │
│      │              │ حبيبي والله اني فخوره فيه وفخوره اني سعوديه الحمدلله دايم وابدا        │          │
├──────┼──────────────┼────────────────────────────────────────────────────────────────────────┼──────────┤
│ 10   │ 2026-02-28   │ نفس افكاري والله                                                       │        2 │
└──────┴──────────────┴────────────────────────────────────────────────────────────────────────┴──────────┘


... and 132 more posts

Analyzing personality...
Trying gemini-2.5-flash...
  Bad response, retrying...


╭──────────────────────────────────────────╮
│   Personality Report — @meowiestgurl90   │
╰──────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ A fiercely passionate and expressive Saudi woman who navigates life with intense emotions, deep faith, and a    │
│ strong sense of national pride, often swinging between profound joy and overwhelming frustration.               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────────── Emotions ────────────────────────────────────────────────────╮
│ Dominant: Love, Frustration, Anxiety, Anger, Sadness, Joy                                                       │
│ Range: Wide                                                                                                     │
│ Positivity: 40%                                                                                                 │
│                                                                                                                 │
│ Experiences emotions intensely and expresses them openly. Prone to stress, anxiety, and frustration, often      │
│ feeling overwhelmed by circumstances. However, also capable of profound joy, affection, and enthusiasm for      │
│ things she loves.                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────────── Social ─────────────────────────────────────────────────────╮
│ Style: Direct, expressive, seeks connection and engagement. Can be confrontational or critical when provoked or │
│ when her values are challenged, but also very supportive and caring towards her in-group.                       │
│ Influence: Active participant/Content creator                                                                   │
│ Engagement: High                                                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────────── Details ────────────────────────────────────────────────────╮
│   Values              Faith (Islam) · Family · National Pride (Saudi Arabia) · Authenticity/Honesty ·           │
│                       Respect/Good Manners · Loyalty                                                            │
│   Strengths           Passionate · Expressive · Loyal · Faithful · Resilient · Socially Engaging                │
│   Growth Areas        Emotional Regulation (managing anger and frustration) · Patience/Tolerance (with others   │
│                       and processes) · Consistency in Personal Goals · Managing Anxiety and Overthinking        │
│   Fun Facts           She intensely loves Ramadan food, especially Vimto and soup mixed with sambosa. · She     │
│                       once confused eye drops with vitamin pills for eyes, leading to a humorous                │
│                       misunderstanding. · She dreams about her Twitter follower count, indicating her           │
│                       engagement with the platform. · She is named روان (Rawan) and has a private Instagram     │
│                       account specifically for women. · She believes she might sometimes be living in a         │
│                       parallel universe due to unusual feelings.                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Generated by Gemini AI · not a clinical assessment

Saved to meowiestgurl90_personality.json
